# Notebook 2: Training Your First RL Agent - DQN for Intraday Trading

## Welcome Back! 🎯

In Notebook 1, you learned about the data. Now it's time to train your first **Reinforcement Learning (RL) agent**!

### What you'll learn:
1. **What is Reinforcement Learning?** (Simple explanation)
2. **What is DQN?** (Deep Q-Network)
3. **Setting up a trading environment**
4. **Training a DQN agent on intraday data**
5. **Evaluating trading performance**
6. **Visualizing results**

### What is RL in Trading?
Think of RL like teaching a robot to trade:
- **Agent** = Your trading bot
- **Environment** = The market
- **State** = Current market conditions (prices, indicators)
- **Actions** = Buy, Hold, or Sell
- **Reward** = Profit or loss from your actions

The agent learns by trial and error to maximize rewards (profits)! 🚀

## Step 1: Import Libraries and Setup

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add TradeMaster to path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(ROOT)

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from mmcv import Config

# TradeMaster imports
from trademaster.utils import replace_cfg_vals, set_seed
from trademaster.nets.builder import build_net
from trademaster.environments.builder import build_environment
from trademaster.datasets.builder import build_dataset
from trademaster.agents.builder import build_agent
from trademaster.optimizers.builder import build_optimizer
from trademaster.losses.builder import build_loss
from trademaster.trainers.builder import build_trainer
from trademaster.transition.builder import build_transition

# Set random seed for reproducibility
set_seed(42)

print("✅ All libraries imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎯 CUDA available: {torch.cuda.is_available()}")

## Step 2: Configure the Trading Environment

Let's set up our configuration for training. We'll use **Micro Crude Oil (MCL-1m)** as an example.

In [ ]:
# Choose your contract (change this to try different contracts!)
CONTRACT = 'MCL-1m'  # Options: 'MCL-1m', 'MGC-1m', 'mes-1m', 'ng', 'si'

# Define paths
DATA_DIR = Path('./data') / CONTRACT
WORK_DIR = Path('./saved_models') / CONTRACT
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"🎯 Selected Contract: {CONTRACT}")
print(f"📁 Data directory: {DATA_DIR}")
print(f"💾 Models will be saved to: {WORK_DIR}")

## Step 3: Create Configuration

We'll create a configuration that tells the RL agent:
- What data to use
- What features (technical indicators) to look at
- How to learn (learning rate, batch size, etc.)
- What actions it can take

In [ ]:
# Create configuration dictionary
config = {
    # Task and dataset info
    'task_name': 'algorithmic_trading',
    'dataset_name': CONTRACT,
    'work_dir': str(WORK_DIR),
    
    # Data configuration
    'data': {
        'type': 'AlgorithmicTradingDataset',
        'data_path': str(DATA_DIR),
        'train_path': str(DATA_DIR / 'train.csv'),
        'valid_path': str(DATA_DIR / 'valid.csv'),
        'test_path': str(DATA_DIR / 'test.csv'),
        'tech_indicator_list': [
            'high', 'low', 'open', 'close', 'adjcp',
            'zopen', 'zhigh', 'zlow', 'zadjcp', 'zclose',
            'zd_5', 'zd_10', 'zd_15', 'zd_20', 'zd_25', 'zd_30'
        ],
        'backward_num_day': 5,  # Look back 5 time periods
        'forward_num_day': 5,   # Look forward 5 time periods
        'test_dynamic': '-1'
    },
    
    # Environment configuration
    'environment': {
        'type': 'AlgorithmicTradingEnvironment'
    },
    
    # DQN Agent configuration
    'agent': {
        'type': 'AlgorithmicTradingDQN',
        'max_step': 10000,
        'reward_scale': 1,
        'repeat_times': 1,
        'gamma': 0.9,  # Discount factor (how much to value future rewards)
        'batch_size': 64,
        'clip_grad_norm': 3.0,
        'soft_update_tau': 0,
        'state_value_tau': 0.005
    },
    
    # Training configuration
    'trainer': {
        'type': 'AlgorithmicTradingTrainer',
        'epochs': 5,  # Start with 5 epochs (increase for better results)
        'work_dir': str(WORK_DIR),
        'seeds_list': (42,),
        'batch_size': 64,
        'horizon_len': 128,
        'buffer_size': 100000,
        'num_threads': 4,
        'if_remove': False,
        'if_discrete': True,
        'if_off_policy': True,
        'if_keep_save': True,
        'if_over_write': False,
        'if_save_buffer': False
    },
    
    # Loss function
    'loss': {'type': 'MSELoss'},
    
    # Optimizer
    'optimizer': {'type': 'Adam', 'lr': 0.001},
    
    # Q-Network (the brain of our agent)
    'act': {
        'type': 'QNet',
        'state_dim': 82,  # 16 features * 5 periods + 2 (cash, position)
        'action_dim': 3,  # 0: Sell, 1: Hold, 2: Buy
        'dims': (64, 32),  # Hidden layer sizes
        'explore_rate': 0.25  # Exploration rate (25% random actions)
    },
    
    'cri': None,
    
    # Transition
    'transition': {'type': 'Transition'},
    
    'batch_size': 64
}

# Convert to mmcv Config object
cfg = Config(config)
cfg = replace_cfg_vals(cfg)

print("✅ Configuration created successfully!")
print(f"\n📊 Key Parameters:")
print(f"  - Epochs: {cfg.trainer['epochs']}")
print(f"  - Batch size: {cfg.batch_size}")
print(f"  - Learning rate: {cfg.optimizer['lr']}")
print(f"  - Actions: 3 (Sell, Hold, Buy)")
print(f"  - Technical indicators: {len(cfg.data['tech_indicator_list'])}")

## Step 4: Build Components

Now let's build all the components we need for training:
1. **Dataset** - Loads and prepares the data
2. **Environment** - Simulates the trading market
3. **Network** - The neural network (brain of the agent)
4. **Agent** - The DQN algorithm
5. **Trainer** - Manages the training process

In [ ]:
print("🔨 Building components...\n")

# 1. Build dataset
print("📊 Loading dataset...")
dataset = build_dataset(cfg)
print(f"   ✅ Dataset loaded: {len(dataset.train_df):,} training samples")

# 2. Build environment
print("\n🌍 Creating trading environment...")
train_env = build_environment(cfg, task='train')
valid_env = build_environment(cfg, task='valid')
print(f"   ✅ Environments created")
print(f"   - State dimension: {train_env.state_dim}")
print(f"   - Action dimension: {train_env.action_dim}")
print(f"   - Initial capital: ${train_env.initial_amount:,.0f}")

# 3. Build neural network (Q-Network)
print("\n🧠 Building Q-Network...")
act = build_net(cfg.act)
print(f"   ✅ Network created with {sum(p.numel() for p in act.parameters()):,} parameters")

# 4. Build optimizer and loss
print("\n⚙️ Setting up optimizer and loss...")
act_optimizer = build_optimizer(cfg, act)
criterion = build_loss(cfg)
print(f"   ✅ Optimizer: {cfg.optimizer['type']}")
print(f"   ✅ Loss function: {cfg.loss['type']}")

# 5. Build transition (for experience replay)
print("\n💾 Setting up experience replay...")
transition = build_transition(cfg)
print(f"   ✅ Transition buffer ready")

# 6. Build agent
print("\n🤖 Creating DQN agent...")
agent = build_agent(cfg, train_env, act, None, act_optimizer, None, criterion)
print(f"   ✅ Agent created and ready to learn!")

# 7. Build trainer
print("\n🎓 Setting up trainer...")
trainer = build_trainer(cfg, train_env, valid_env, agent, transition)
print(f"   ✅ Trainer ready!")

print("\n" + "="*60)
print("✅ All components built successfully!")
print("="*60)

## Step 5: Train the Agent! 🚀

This is where the magic happens! The agent will:
1. Observe market conditions (state)
2. Choose an action (Buy/Hold/Sell)
3. Get a reward (profit/loss)
4. Learn from experience
5. Repeat thousands of times!

**Note**: Training can take 10-30 minutes depending on your hardware. Grab a coffee! ☕

In [ ]:
print("🚀 Starting training...\n")
print("This may take 10-30 minutes. The agent is learning to trade!")
print("\n" + "="*60)

# Start training
trainer.train_and_valid()

print("\n" + "="*60)
print("🎉 Training complete!")
print("="*60)

## Step 6: Evaluate Performance

Let's test our trained agent on unseen data and see how well it trades!

In [ ]:
print("📈 Evaluating agent on test data...\n")

# Build test environment
test_env = build_environment(cfg, task='test')

# Test the agent
trainer.test()

print("\n✅ Evaluation complete!")

## Step 7: Visualize Trading Results

Let's visualize how our agent performed during testing!

In [ ]:
# Load test results
result_path = WORK_DIR / 'test' / f'{CONTRACT}_algorithmic_trading_test.csv'

if result_path.exists():
    results_df = pd.read_csv(result_path)
    
    # Create visualization
    fig, axes = plt.subplots(3, 1, figsize=(15, 12))
    
    # Plot 1: Portfolio Value Over Time
    axes[0].plot(results_df.index, results_df['portfolio_value'], linewidth=2, color='green')
    axes[0].axhline(y=test_env.initial_amount, color='red', linestyle='--', label='Initial Capital')
    axes[0].set_title('Portfolio Value Over Time', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Portfolio Value ($)', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Actions Taken
    if 'action' in results_df.columns:
        action_colors = {0: 'red', 1: 'gray', 2: 'green'}
        action_labels = {0: 'Sell', 1: 'Hold', 2: 'Buy'}
        for action, color in action_colors.items():
            mask = results_df['action'] == action
            axes[1].scatter(results_df.index[mask], [action]*mask.sum(), 
                          c=color, alpha=0.6, label=action_labels[action], s=10)
        axes[1].set_title('Trading Actions Over Time', fontsize=14, fontweight='bold')
        axes[1].set_ylabel('Action', fontsize=12)
        axes[1].set_yticks([0, 1, 2])
        axes[1].set_yticklabels(['Sell', 'Hold', 'Buy'])
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Returns
    if 'portfolio_return' in results_df.columns:
        axes[2].plot(results_df.index, results_df['portfolio_return'], linewidth=1, alpha=0.7)
        axes[2].axhline(y=0, color='red', linestyle='--')
        axes[2].set_title('Portfolio Returns', fontsize=14, fontweight='bold')
        axes[2].set_xlabel('Time Step', fontsize=12)
        axes[2].set_ylabel('Return', fontsize=12)
        axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\n📊 Trading Performance Summary:\n")
    print("="*60)
    final_value = results_df['portfolio_value'].iloc[-1]
    initial_value = test_env.initial_amount
    total_return = (final_value - initial_value) / initial_value * 100
    
    print(f"Initial Capital:      ${initial_value:,.2f}")
    print(f"Final Portfolio Value: ${final_value:,.2f}")
    print(f"Total Return:         {total_return:+.2f}%")
    print(f"Profit/Loss:          ${final_value - initial_value:+,.2f}")
    
    if 'portfolio_return' in results_df.columns:
        returns = results_df['portfolio_return'].dropna()
        if len(returns) > 0:
            sharpe = returns.mean() / (returns.std() + 1e-8) * np.sqrt(252 * 1440)
            print(f"\nSharpe Ratio (Ann):   {sharpe:.2f}")
            print(f"Max Drawdown:         {(results_df['portfolio_value'].min() - initial_value) / initial_value * 100:.2f}%")
    
    print("="*60)
    
else:
    print(f"⚠️  Results file not found at {result_path}")

## Step 8: Action Distribution Analysis

Let's see what actions our agent learned to take:

In [ ]:
if result_path.exists() and 'action' in results_df.columns:
    # Count actions
    action_counts = results_df['action'].value_counts().sort_index()
    action_labels = {0: 'Sell', 1: 'Hold', 2: 'Buy'}
    
    # Create pie chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Pie chart
    colors = ['red', 'gray', 'green']
    ax1.pie(action_counts.values, labels=[action_labels[i] for i in action_counts.index],
            autopct='%1.1f%%', colors=colors, startangle=90)
    ax1.set_title('Action Distribution', fontsize=14, fontweight='bold')
    
    # Bar chart
    ax2.bar([action_labels[i] for i in action_counts.index], action_counts.values, color=colors)
    ax2.set_title('Action Counts', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Count', fontsize=12)
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("\n🎯 Action Statistics:")
    print("="*40)
    for action, count in action_counts.items():
        percentage = count / len(results_df) * 100
        print(f"{action_labels[action]:6s}: {count:5d} times ({percentage:5.1f}%)")
    print("="*40)

## Summary and Next Steps

### 🎉 Congratulations!

You've successfully trained your first RL trading agent using DQN! Here's what you accomplished:

1. ✅ **Understood RL basics** (agent, environment, state, action, reward)
2. ✅ **Configured a DQN agent** for intraday trading
3. ✅ **Trained the agent** on real market data
4. ✅ **Evaluated performance** on test data
5. ✅ **Visualized results** and analyzed actions

### 🧠 What is DQN?
**Deep Q-Network (DQN)** uses a neural network to learn the "quality" (Q-value) of taking each action in each state. It learns by:
- Trying different actions
- Getting rewards (profit/loss)
- Updating the network to predict better actions
- Exploring vs exploiting (25% random, 75% learned)

### 💡 Key Insights:
- DQN works well for **discrete actions** (Buy/Hold/Sell)
- Training takes time but agents get smarter!
- The agent learned patterns from historical data
- More training epochs = better performance (usually)

### 🔧 Experiment Ideas:
1. **Change the contract**: Try 'MGC-1m', 'mes-1m', 'ng', or 'si'
2. **Increase epochs**: Change from 5 to 20 for better learning
3. **Adjust learning rate**: Try 0.0001 or 0.01
4. **Modify exploration**: Change explore_rate from 0.25

### 🚀 Next Steps:
In **Notebook 3**, you'll learn:
- **PPO (Proximal Policy Optimization)** - More stable than DQN
- Policy-based vs value-based RL
- Better exploration strategies
- Continuous action spaces

Ready for the next algorithm? Let's go! 🎯

---
**Pro Tips**: 
- Save your trained models in `saved_models/` folder
- Try different contracts to see which ones are easier to trade
- Monitor the training loss - it should decrease over time
- Compare DQN with other algorithms in later notebooks! 💡